<a href="https://colab.research.google.com/github/ABROCK1516/Ukraine-Losses-Weather/blob/main/Ukraine_Losses_Weather.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Russian Equipment & Personnel Losses & Front-line Weather**
## *Data Acquisition and Preprocessing – Individual project*
**Alex Brock | C23426432 | TU912/4**

## **Introduction**
This project combines daily figures for Russian equipment and personnel losses in the war in Ukraine with weather observations from near the front line. Weather is often said to shape the fighting, for example mud seasons slowing armoured vehicles, or wind and fog grounding FPV drones which allows for more ground assaults and offensive actions. This project tests whether those patterns can be seen in the data.

I chose this topic because I have been following the war closely since the full scale invasion of Ukraine in 2022. I keep up to date with the war through YouTube channels and the news. From the start of 2023 onwards, I have watched and read about both sides of the war using weather patterns and conditions to their advantage such as fog to aid in assaulting dug-in positions.

I was particularly interested in the military side: how weapons and drones are used, and how much outside conditions like the weather affect them. Commentary on the war often claims that weather changes the fighting, and I wanted to see whether the data actually supports that.

## **Research Questions**
The project is built in three layers, each adding a dataset and new questions:

1. **Kaggle data alone:** How have daily losses changed over the course of the war? Has the mix of equipment lost changed, and has the ratio of personnel to vehicles lost changed over time?
2. **Adding weather:** Do daily losses vary with weather conditions such as rain, freezing temperatures, wind and low visibility? Which loss categories are most affected?
3. **Adding combat engagements (extension):** When losses change with the weather, is that because there is less fighting, or because the fighting causes fewer losses?

## **Data Sources**

| Source | What it contains | Period | How it was obtained | Role |
|---|---|---|---|---|
| [Kaggle: 2022 Russia Ukraine War (piterfm)](https://www.kaggle.com/datasets/piterfm/2022-ukraine-russian-war) | Daily cumulative Russian equipment losses by category, and personnel losses | 25 Feb 2022 – 10 Oct 2026 (1,689 rows) | Downloaded as two CSV files | Main dataset |
| [pogodaiklimat.ru](http://www.pogodaiklimat.ru/) | 3-hourly weather observations: wind, visibility, cloud cover, weather phenomena, temperature, precipitation | Jan 2023 – Oct 2026 | Web scraping (urllib and BeautifulSoup) | Complementary dataset |
| [ukraine.vefgreining.com](https://ukraine.vefgreining.com/) | Daily number of combat engagements reported by Ukraine's General Staff | 22 Feb 2023 – 16 Oct 2026 | JSON data file | Extension |

Weather is taken from two stations: **occupied Donetsk** (station 34519), which is close to most of the heaviest fighting since 2023, and **Kharkiv** (station 34300), in the Kharkiv region, where fighting continues around Kupiansk, it is used as a second station to check whether results hold.

## **Notes on the data**
- **These are claimed figures.** The loss and engagement figures are reported by Ukraine and have not been independently verified.
- **Personnel figures are estimates.** The dataset itself marks them as approximate (“about” or “more”).
- **"Occupied Donetsk".** The Donetsk weather station is in Russian-occupied territory, and the weather site lists it under Russia.
- **Weather coverage.** The weather site has little data for 2022 (spot-checked in Step 2: both stations have no readings in February or July 2022, and Donetsk only resumes on 18 November 2022, with about 18% of readings missing), so the weather analysis covers 2023 onwards. The Kaggle data is kept in full for Step 1
- **One weather point for a long front.** The front line is over 1,000 km long, so weather at one station is only an approximation of conditions where the fighting took place.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Import the os library, which lets Python work with folders and file paths
import os
os.chdir("/content/drive/MyDrive/Data ACQ - Project/Data")
!pwd

/content/drive/MyDrive/Data ACQ - Project/Data


## **Step 1: Import the Kaggle dataset**
The dataset is "2022 Russia Ukraine War" by piterfm on Kaggle. It has two CSV files:
running totals of Russian equipment losses and personnel losses.

**About the dataset**

**Goal:** load the Kaggle data, understand its structure, columns, types and values, and turn the running totals into daily losses.

**Assignment:** Step 1, importing and assessing the Kaggle dataset **(20 marks).**

**Research link:** daily losses are the main measure in all three research questions.

The main dataset is ["2022 Russia Ukraine War" by piterfm](https://www.kaggle.com/datasets/piterfm/2022-ukraine-russian-war) on Kaggle. It records Russian losses in the war as reported by official Ukrainian sources, in two CSV files:
- `russia_losses_equipment.csv` (1,689 rows × 20 columns): losses by equipment category, such as tanks, armoured personnel carriers (APCs), artillery and drones.
- `russia_losses_personnel.csv` (1,689 rows × 5 columns): personnel losses and prisoners of war (POW).

Both files cover 25 February 2022 to 10 October 2026, with one row per day.

The figures are **cumulative**: each row gives the total lost since the start of the invasion, not the losses on that day, so daily losses have to be calculated. The data was downloaded on 16 October 2026. The dataset is updated daily, so it is not downloaded again, which keeps the results reproducible.

### **1.1 - Loading the Two Files**
I load both CSV files into pandas DataFrames, one called `equipment` and one called `personnel`.

In [3]:
# Import pandas, the library used to load and work with tables of data
import pandas as pd

# Load the equipment losses CSV into a DataFrame (a table) called equipment
equipment = pd.read_csv("russia_losses_equipment.csv")

# Load the personnel losses CSV into a DataFrame called personnel
personnel = pd.read_csv("russia_losses_personnel.csv")

### **1.2 - First look: The Equipment File**
**Four checks:** The size and first rows, the data type of each column, summary statistics, and the number of missing values in each column.

In [4]:
print(equipment.shape)  # size of the table: (rows, columns)
equipment.head(10)  # first 10 rows, to see what the data looks like

(1689, 20)


,date,day,aircraft,helicopter,tank,APC,field artillery,MRL,military auto,fuel tank,drone,naval ship,anti-aircraft warfare,special equipment,mobile SRBM system,greatest losses direction,vehicles and fuel tanks,cruise missiles,submarines,ground robotic systems
0,2026-10-10,1690,443,356,12406,25515,51000,2201,NaN,NaN,556926,35,1699,4812.0,NaN,NaN,160946.0,5126.0,2.0,2917.0
1,2026-10-09,1689,443,356,12402,25509,50950,2199,NaN,NaN,555335,35,1693,4810.0,NaN,NaN,160412.0,5126.0,2.0,2908.0
2,2026-10-08,1688,443,356,12399,25502,50883,2194,NaN,NaN,553578,35,1692,4806.0,NaN,NaN,159795.0,5126.0,2.0,2890.0
3,2026-10-07,1687,443,356,12396,25498,50836,2192,NaN,NaN,551876,35,1690,4800.0,NaN,NaN,159128.0,5126.0,2.0,2878.0
4,2026-10-06,1686,443,356,12396,25494,50785,2184,NaN,NaN,549986,35,1688,4795.0,NaN,NaN,158551.0,5126.0,2.0,2866.0
5,2026-10-05,1685,443,356,12393,25487,50733,2182,NaN,NaN,548095,35,1688,4793.0,NaN,NaN,157985.0,5126.0,2.0,2859.0
6,2026-10-04,1684,443,356,12391,25483,50686,2179,NaN,NaN,546240,35,1686,4788.0,NaN,NaN,157258.0,5126.0,2.0,2837.0
7,2026-10-03,1683,443,356,12391,25477,50637,2178,NaN,NaN,544498,35,1683,4783.0,NaN,NaN,156731.0,5126.0,2.0,2824.0
8,2026-10-02,1682,443,356,12388,25471,50592,2175,NaN,NaN,542853,35,1683,4780.0,NaN,NaN,156309.0,5126.0,2.0,2810.0
9,2026-10-01,1681,443,356,12388,25464,50555,2174,NaN,NaN,541383,35,1683,4774.0,NaN,NaN,155887.0,5126.0,2.0,2803.0


In [5]:
# Show the column names and the data type of each column
equipment.dtypes

,0
date,object
day,int64
aircraft,int64
helicopter,int64
tank,int64
APC,int64
field artillery,int64
MRL,int64
military auto,float64
fuel tank,float64


In [6]:
# Summary statistics for each numeric column:
equipment.describe()

,day,aircraft,helicopter,tank,APC,field artillery,MRL,military auto,fuel tank,drone,naval ship,anti-aircraft warfare,special equipment,mobile SRBM system,vehicles and fuel tanks,cruise missiles,submarines,ground robotic systems
count,1689.000000,1689.000000,1689.000000,1689.000000,1689.000000,1689.000000,1689.000000,65.000000,65.000000,1689.000000,1689.000000,1689.000000,1670.000000,36.000000,1624.000000,1624.000000,1123.000000,161.000000
mean,846.000000,343.914742,303.917703,7302.714624,14825.561279,17909.660746,1046.749556,1047.507692,69.323077,66177.216104,23.708111,788.043813,2282.670060,3.944444,37143.435961,2428.223522,1.266251,2036.956522
std,487.716619,79.985283,60.524649,3836.696267,7956.872953,15393.569483,548.811027,466.162060,7.545917,125776.097509,7.141723,473.888851,1695.212102,0.333333,38493.183090,1501.023468,0.442194,475.056514
min,2.000000,10.000000,7.000000,80.000000,516.000000,49.000000,4.000000,100.000000,60.000000,0.000000,2.000000,0.000000,10.000000,2.000000,1796.000000,84.000000,1.000000,1306.000000
25%,424.000000,308.000000,294.000000,3675.000000,7131.000000,2837.000000,539.000000,600.000000,60.000000,2402.000000,18.000000,289.000000,381.000000,4.000000,6452.750000,1181.250000,1.000000,1636.000000
50%,846.000000,359.000000,326.000000,7974.000000,15307.000000,13959.000000,1104.000000,1178.000000,73.000000,11187.000000,28.000000,853.000000,2425.500000,4.000000,21002.000000,2401.000000,1.000000,1981.000000
75%,1268.000000,421.000000,340.000000,11104.000000,23130.000000,31458.000000,1466.000000,1437.000000,76.000000,51043.000000,28.000000,1207.000000,3942.750000,4.000000,60242.750000,3635.500000,2.000000,2442.000000
max,1690.000000,443.000000,356.000000,12406.000000,25515.000000,51000.000000,2201.000000,1701.000000,76.000000,556926.000000,35.000000,1699.000000,4812.000000,4.000000,160946.000000,5126.000000,2.000000,2917.000000


In [7]:
# Count the missing (empty) values in each column
equipment.isnull().sum()

,0
date,0
day,0
aircraft,0
helicopter,0
tank,0
APC,0
field artillery,0
MRL,0
military auto,1624
fuel tank,1624


### **1.3 - First Look: The Personnel File**
**Four checks:** The size and first rows, the data type of each column, summary statistics, and the number of missing values in each column.

In [8]:
print(personnel.shape)  # size of the table: (rows, columns)
personnel.head()  # first 5 rows

(1689, 5)


,date,day,personnel,personnel*,POW
0,2026-10-10,1690,1550910,about,NaN
1,2026-10-09,1689,1549490,about,NaN
2,2026-10-08,1688,1547780,about,NaN
3,2026-10-07,1687,1545950,about,NaN
4,2026-10-06,1686,1544260,about,NaN


In [9]:
# Show the column names and the data type of each column
personnel.dtypes

,0
date,object
day,int64
personnel,int64
personnel*,object
POW,float64


In [10]:
# Summary statistics for each numeric column:
personnel.describe()

,day,personnel,POW
count,1689.000000,1.689000e+03,62.000000
mean,846.000000,6.250922e+05,386.387097
std,487.716619,4.793099e+05,131.440363
min,2.000000,2.800000e+03,0.000000
25%,424.000000,1.864200e+05,389.000000
50%,846.000000,5.286200e+05,421.000000
75%,1268.000000,1.067100e+06,474.500000
max,1690.000000,1.550910e+06,496.000000


In [11]:
# Count the missing (empty) values in each column
personnel.isnull().sum()

,0
date,0
day,0
personnel,0
personnel*,0
POW,1627


### **1.4 - Do the Two Files Line up?**
The two files will be joined on `date`, so I check they have the same date in every row. `.all()` returns `True` only if every row matches.

In [12]:
# Compare the date columns row by row; .all() is True only if every row matches
(equipment["date"] == personnel["date"]).all()

np.True_

### **1.5 - When was Each Category Reported?**
Several columns have many missing values. For each one, I print the first and last date with data and the number of filled rows, to see whether the gaps are errors or changes in what was reported. I do the same for `POW` in the personnel file.

In [13]:
# Columns with missing values, to check when each one has data
columns_to_check = ["military auto", "fuel tank", "vehicles and fuel tanks",
                    "cruise missiles", "submarines", "ground robotic systems",
                    "mobile SRBM system", "greatest losses direction",
                    "special equipment"]  # special equipment has 19 missing values

for col in columns_to_check:  # go through each column name in the list
    filled = equipment[equipment[col].notnull()]  # keep only the rows where this column is NOT empty
    print(col, "|", filled["date"].min(), "to", filled["date"].max(), "|", len(filled), "rows")  # first date, last date, rows with data

military auto | 2022-02-25 to 2022-04-30 | 65 rows
fuel tank | 2022-02-25 to 2022-04-30 | 65 rows
vehicles and fuel tanks | 2022-05-01 to 2026-10-10 | 1624 rows
cruise missiles | 2022-05-01 to 2026-10-10 | 1624 rows
submarines | 2023-09-14 to 2026-10-10 | 1123 rows
ground robotic systems | 2026-05-03 to 2026-10-10 | 161 rows
mobile SRBM system | 2022-03-26 to 2022-04-30 | 36 rows
greatest losses direction | 2022-04-25 to 2023-01-02 | 203 rows
special equipment | 2022-03-16 to 2026-10-10 | 1670 rows


In [14]:
# Keep only the rows where POW has a value
pow_filled = personnel[personnel["POW"].notnull()]
# Print the first date, last date and number of rows with a POW value
print(pow_filled["date"].min(), "to", pow_filled["date"].max(), "|", len(pow_filled), "rows")

2022-02-25 to 2022-04-27 | 62 rows


### **1.6 - Observaions on the Kaggle Dataset**
**Structure:** Each row is one day, and the rows are in newest-first order (the first row is 10 October 2026). The figures are running totals since the start of the invasion, not the losses on that day. The equipment file has 1,689 rows and 20 columns, and the personnel file has 1,689 rows and 5 columns. Both files have the same date in every row (the check returned `True`), so they can be joined on `date`.

**Data types:** `date` is stored as `object`, which means pandas treats it as text, so it needs converting before it can be sorted or compared as a date. Some whole-number columns are stored as decimals (`float64`) because they contain empty values, which a whole-number column can't hold.

**Values:**
- `day` starts at 2. The invasion began on **24 February 2022 (day 1)**, so the first row (25 February) already covers two days of losses.
- Because the figures are running totals, each **column's minimum is just its first value**. For example, 516 APCs had already been claimed in the first row.
- `drone` and `anti-aircraft warfare` start at 0. Drones were used from the start of the invasion, so these categories probably **weren't being counted** yet.
- Some categories start high when they first appear: `cruise missiles` at 84 and `ground robotic systems` at 1,306. This suggests earlier losses were added in one go on the first day. Since ground robotic systems were first reported (3 May 2026).

**Missing values:** The gaps aren't errors. They come from changes in which categories the daily report included:
- **Replaced:** `military auto` and `fuel tank` stop on 30 April 2022, and `vehicles and fuel tanks` starts the next day. Together they fill every row (65 + 1,624 = 1,689), so the **two old categories were combined into one**.
- **Added later:** `special equipment` (16 March 2022: its 19 missing values are the first 19 days of the war), `cruise missiles` (1 May 2022), `submarines` (14 September 2023) and `ground robotic systems` (3 May 2026). Submarines first appear the day after Ukraine's missile strike on the submarine *Rostov-on-Don* in Sevastopol on 13 September 2023 ([Naval News](https://www.navalnews.com/?p=58386)). This fits each day's report covering the day before.
- **Dropped:** `mobile SRBM system` was only reported from 26 March to 30 April 2022. `greatest losses direction` contains place names, not numbers, and was only reported from 25 April 2022 to 2 January 2023, with gaps, so it is not used.
- **Personnel:** `POW` was only reported from 25 February to 27 April 2022. After that it stopped being reported; the data doesn't say why - **possibly the data was confidential.**

So each category can only be compared over the period it was reported.



### **1.7 From Running Totals to Daily Losses**
The figures are running totals, but the research questions are about losses on each day. To get daily losses, I:

**1.** Convert `date` from text to real dates, so pandas can sort and compare them

**2.** Join the equipment and personnel files into one table on `date`

**3.** Sort the rows from oldest to newest (the files are newest first)

**4.** Subtract the previous day's total from each day's total

#### **1.7.1 - Converting the dates**

In [15]:
equipment["date"] = pd.to_datetime(equipment["date"])  # convert equipment dates from text to real dates
personnel["date"] = pd.to_datetime(personnel["date"])  # same for personnel

print(equipment["date"].dtype)  # should now show datetime64 instead of object
print(personnel["date"].dtype)  # same check for personnel

datetime64[ns]
datetime64[ns]


#### **1.7.2 - Joining the Two Files**

Both files have the same 1,689 dates **(checked in 1.4)**, so I join them on `date` with `pd.merge`. I use `how="outer"`, which keeps dates found in either file, so any mismatch would show up as empty values instead of being **silently dropped**. I leave out the personnel file's `day` column, because the equipment file already has it.

In [16]:
personnel_cols = personnel[["date", "personnel", "personnel*", "POW"]]  # keep only the columns needed (double brackets = a list of columns)

losses = pd.merge(equipment, personnel_cols, on="date", how="outer")  # join on date; "outer" keeps dates found in either file, so mismatches would show as empty values

losses.shape  # expect 1,689 rows and 23 columns (20 from equipment + 3 from personnel)

(1689, 23)

#### **1.7.3 - Sorting by Date**
`.sort_values("date")` puts the rows in order from oldest to newest, so each row follows the day before it. `.reset_index(drop=True)` renumbers the rows 0, 1, 2… in the new order.

In [17]:
losses = losses.sort_values("date")  # oldest first, so each row follows the day before it
losses = losses.reset_index(drop=True)  # renumber the rows; drop=True throws away the old row numbers

losses.head()  # the first row should now be 25 Feb 2022

,date,day,aircraft,helicopter,tank,APC,field artillery,MRL,military auto,fuel tank,...,special equipment,mobile SRBM system,greatest losses direction,vehicles and fuel tanks,cruise missiles,submarines,ground robotic systems,personnel,personnel*,POW
0,2022-02-25,2,10,7,80,516,49,4,100.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2800,about,0.0
1,2022-02-26,3,27,26,146,706,49,4,130.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4300,about,0.0
2,2022-02-27,4,27,26,150,706,50,4,130.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4500,about,0.0
3,2022-02-28,5,29,29,150,816,74,21,291.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5300,about,0.0
4,2022-03-01,6,29,29,198,846,77,24,305.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5710,about,200.0


#### **Checking for Missing Days**
`.diff()` on the dates gives the gap between each date and the one before it, and `.value_counts()` counts how often each gap size appears. If no days are missing or repeated, every gap is 1 day.

In [18]:
losses["date"].diff().value_counts()  # gap between each date and the one before; every gap should be 1 day

,count
date,
1 days,1688


#### **1.7.4 - Calculating daily losses**

In [19]:
# The running-total columns to turn into daily losses
loss_columns = ["aircraft", "helicopter", "tank", "APC", "field artillery", "MRL",
                "drone", "naval ship", "anti-aircraft warfare", "special equipment",
                "vehicles and fuel tanks", "cruise missiles", "submarines",
                "ground robotic systems", "personnel"]

for col in loss_columns:  # go through each column in the list
    losses["daily " + col] = losses[col].diff()  # today's total minus yesterday's = losses claimed that day

losses[["date", "tank", "daily tank", "personnel", "daily personnel"]].head(10)  # check a few columns side by side

,date,tank,daily tank,personnel,daily personnel
0,2022-02-25,80,NaN,2800,NaN
1,2022-02-26,146,66.0,4300,1500.0
2,2022-02-27,150,4.0,4500,200.0
3,2022-02-28,150,0.0,5300,800.0
4,2022-03-01,198,48.0,5710,410.0
5,2022-03-02,211,13.0,5840,130.0
6,2022-03-03,217,6.0,9000,3160.0
7,2022-03-04,251,34.0,9166,166.0
8,2022-03-05,269,18.0,10000,834.0
9,2022-03-06,285,16.0,11000,1000.0


### **1.8 - Checking the Daily Figures**
A running total should never go down, so a negative daily figure would mean the source corrected an earlier number.

In [20]:
for col in loss_columns:  # go through each loss category
    negatives = (losses["daily " + col] < 0).sum()  # count the days where the daily figure is below zero
    print(col, negatives)  # print the category and its count

aircraft 2
helicopter 0
tank 1
APC 1
field artillery 1
MRL 2
drone 0
naval ship 0
anti-aircraft warfare 2
special equipment 2
vehicles and fuel tanks 0
cruise missiles 2
submarines 0
ground robotic systems 0
personnel 0


#### **Summary Statistics for the Main Categories**
`.describe()` shows the typical daily loss (the median, in the 50% row) and the range for the categories used most in the analysis.

In [21]:
# Summary statistics for the daily figures of the main categories
losses[["daily tank", "daily APC", "daily field artillery", "daily drone", "daily personnel"]].describe()

,daily tank,daily APC,daily field artillery,daily drone,daily personnel
count,1688.000000,1688.000000,1688.000000,1688.000000,1688.000000
mean,7.302133,14.809834,30.184242,329.932464,917.126777
std,6.535839,13.564484,22.665864,626.922771,420.982929
min,-2.000000,-14.000000,-10.000000,0.000000,0.000000
25%,3.000000,5.000000,10.000000,8.000000,600.000000
50%,5.000000,11.000000,26.000000,36.000000,970.000000
75%,10.000000,20.000000,46.000000,205.250000,1210.000000
max,66.000000,190.000000,122.000000,2719.000000,3160.000000


#### **The Negative and Zero Days**
For the three main categories with a negative minimum, I list the dates of the negative days. I also list the days where no personnel losses were added.

In [22]:
for col in ["tank", "APC", "field artillery"]:  # the columns with negative minimums
    neg_days = losses[losses["daily " + col] < 0]  # keep only the rows where the daily figure is negative
    print(neg_days[["date", "daily " + col]])  # show the date and the negative value

zero_personnel = losses[losses["daily personnel"] == 0]  # rows where no personnel losses were added
print(zero_personnel[["date", "personnel", "daily personnel"]])  # show when they happened

          date  daily tank
457 2023-05-28        -2.0
          date  daily APC
230 2022-10-13      -14.0
           date  daily field artillery
1468 2026-03-04                  -10.0
         date  personnel  daily personnel
10 2022-03-07      11000              0.0
12 2022-03-09      12000              0.0
13 2022-03-10      12000              0.0
14 2022-03-11      12000              0.0
15 2022-03-12      12000              0.0
16 2022-03-13      12000              0.0
17 2022-03-14      12000              0.0


#### **The Tank Correction in Context**
I show three days either side of the −2 tank day, to see whether it cancels out an unusually high day before it.

In [23]:
losses[["date", "tank", "daily tank"]][454:461]  # rows 454 to 460: three days either side of the tank correction

,date,tank,daily tank
454,2023-05-25,3795,3.0
455,2023-05-26,3796,1.0
456,2023-05-27,3799,3.0
457,2023-05-28,3797,-2.0
458,2023-05-29,3801,4.0
459,2023-05-30,3802,1.0
460,2023-05-31,3802,0.0


### **1.9 - Observations on Daily Losses**
**Dates:** After sorting, every gap between dates is exactly one day (1,688
gaps for 1,689 rows), so no days are missing or repeated.

**Negative days:** A running total should never go down, but it does on 13 days across 8 columns. **Aircraft, MRL, anti-aircraft warfare, special equipment and cruise missiles go down twice each, and tank, APC and field artillery once each** (APC −14 on 13 October 2022, tank −2 on 28 May 2023, field artillery −10 on 4 March 2026). These look like **corrections**, where the source lowered a total it had reported earlier. For the tank, the days either side are normal (+3 and +4), so it doesn't look like it is cancelling out one unusually high day. Drones, vehicles and personnel never go down. The negative days are rare and small, but they need handling before the analysis, because a negative daily loss isn't possible.

**Early figures are uneven:** The largest daily tank figure (66, on 26 February 2022) and the largest daily personnel figure (3,160, on 3 March 2022) both come from the first ten days of the war. Seven days in early March 2022 (7 March and 9–14 March) show zero personnel losses, while the totals sit at round numbers (11,000 and 12,000). This suggests the e**arly estimates were rounded and updated every few days**, so early daily figures show when the estimate was updated rather than when the losses happened. This doesn't affect the weather analysis, which starts in January 2023.

**Drones vs tanks:** The median daily drone loss (36) is much higher than for tanks (5), which fits drones being used in far greater numbers and being cheaper to lose. The drone mean (330) is about nine times the median, because a small number of very large days pull it up. Drone use **grew sharply over the war**, so a whole-war average mixes very different periods. When plotting, drones will need a log scale or a rolling average, and the years compared separately.

## **Step 2: Scraping the Weather Data**
**Goal:** scrape 3-hourly weather observations for two stations near the front line, from January 2023 to October 2026, and turn them into one row per day.

**Assignment:** Step 2, scraping a complementary dataset **(40 marks)**.

**Research link:** adds the weather columns needed for **research question 2**: do daily losses vary with the weather?

The data comes from [pogodaiklimat.ru](http://www.pogodaiklimat.ru), a Russian weather archive that publishes weather station readings every 3 hours (8 per day). I scrape two stations: **occupied Donetsk** (34519) and **Kharkiv** (34300).

**This step has eleven parts:**

**1.** Why this site, and how it links to the Kaggle data

**2.** Scraping ethics

**3.** Fetching and inspecting one page

**4.** The bot parameter

**5.** Parsing one month

**6.** Testing before the full scrape

**7.** Scraping every month

**8.** Checking coverage

**9.** Parsing every page

**10.** Cleaning

**11.** One row per day

### **2.1 - Why This Site and How It Links to the Kaggle Data**
**Why this site:** Commentary on the war links the weather to different kinds of fighting: rain and mud slowing tanks and armoured vehicles, frozen ground letting them move again, and wind and fog grounding drones allowing for more troop movement. Testing these needs more than daily averages. This site records, at every 3-hourly reading, precipitation, temperature (with daily minimum and maximum), snow depth, weather phenomena (rain, snow, fog), visibility, wind speed and gusts, and cloud cover.

**New columns:** The Kaggle dataset only records losses. The weather data adds wind speed and gusts, visibility, weather phenomena, cloud cover, temperature and precipitation, so I can ask whether daily losses change with the weather, and which loss categories are most affected.

**How many Kaggle rows it covers:** The weather analysis starts on 1 January 2023 (see the 2022 check in 2.6). From then to 10 October 2026 is 1,379 of the 1,689 Kaggle days (82%), far above the 10% required. Donetsk has no readings on 63 of those days (see 2.8), which leaves about 1,316 days (78%). The exact number of matched rows is counted after the merge in Step 3.

**The common column:** Both datasets have a date, so they can be joined on it directly, with no bridging dataset needed. The weather dates need preparing first:
- the site gives the date as day.month with no year, so the year is added from the file name
- times are in UTC, so they are converted to local time before deciding which day a reading belongs to
- there are 8 readings per day, so they are combined into one row per day, to match Kaggle's one row per day
- Kaggle dates each figure by the day it was reported (the morning after the fighting), so Kaggle dates are shifted back one day before the merge

### **2.2 - Scraping Ethics**
Before requesting any weather data, I check what the site allows: its robots.txt file and its terms of use. I then summarise how the scraper follows them.

#### **Checking robots.txt**
robots.txt tells automated programs which parts of a site they shouldn't access (week 2 ethics). I fetch it with the same `urllib` method used for the weather pages.

In [24]:
import urllib.request  # module from week 2 for fetching web pages

robots_url = "http://www.pogodaiklimat.ru/robots.txt"  # the site's rules for automated programs

request = urllib.request.Request(robots_url)  # build the request
request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request politely
opener = urllib.request.build_opener()  # object that sends requests
response = opener.open(request)  # send the request
robots_text = response.read().decode("utf-8")  # read the bytes and turn them into text

print(robots_text)  # display the rules

User-agent: *
Disallow: /admin/
Disallow: /metar/
Disallow: /photo/
Disallow: /synop/
Disallow: /summary/
Disallow: /usasummary/
Disallow: /extremal.php?
Disallow: /extremal1.php?
Disallow: /dextremal.php?
Disallow: /dextremal1.php?

User-agent: Yandex
Disallow: /admin/
Disallow: /metar/
Disallow: /photo/
Disallow: /synop/
Disallow: /summary/
Disallow: /usasummary/
Disallow: /extremal.php?
Disallow: /extremal1.php?
Disallow: /dextremal.php?
Disallow: /dextremal1.php?
Host: www.pogodaiklimat.ru


#### **Checking the Terms of Use**
I fetch the site's user agreement and print only the clauses that apply to scraping and publishing data (3.1 and 3.5–3.7), rather than the whole page. My translation is in the next cell.

In [25]:
from bs4 import BeautifulSoup # library from week 3 for searching through HTML

agreement_url = "http://www.pogodaiklimat.ru/agreement.php"  # the site's user agreement page

request = urllib.request.Request(agreement_url)  # build the request
request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
response = opener.open(request)  # send it, reusing the opener from the robots.txt cell
agreement_html = response.read().decode("utf-8")  # read the bytes and turn them into text

agreement_soup = BeautifulSoup(agreement_html, "html.parser")  # turn the HTML into a searchable object
agreement_text = agreement_soup.get_text()  # keep only the text, without the HTML tags

start_3_1 = agreement_text.find("3.1 ")  # position where clause 3.1 starts (the space stops it matching 3.10)
end_3_1 = agreement_text.find("3.2 ")  # position where clause 3.2 starts, i.e. where 3.1 ends
print(agreement_text[start_3_1:end_3_1])  # print clause 3.1 only

start_3_5 = agreement_text.find("3.5 ")  # position where clause 3.5 starts
end_3_7 = agreement_text.find("3.8 ")  # position where clause 3.8 starts, i.e. where 3.7 ends
print(agreement_text[start_3_5:end_3_7])  # print clauses 3.5, 3.6 and 3.7

3.1 Сайт предназначен для использования исключительно в личных некоммерческих целях. Пользователь обязан пользоваться Сайтом в порядке, определенном Соглашением.

3.5 Полное или частичное копирование, воспроизведение, распространение, размещение в свободном доступе в сети Интернет текстовых и графических материалов Сайта, в том числе - элементов дизайна, любое их использование в средствах массовой информации и/или в коммерческих целях без предварительного письменного разрешения Администрации Сайта запрещено.
3.6 Запрещено копирование, распространение, размещение в свободном доступе в сети Интернет Данных, находящихся в разделе Саммари, недоступном для Пользователей без предварительной авторизации.
3.7 При размещении в свободном доступе Данных с Сайта в сети Интернет обязательно необходима активная ссылка на источник информации (Источник: сайт «Погода и Климат»).



### **Translation of the Printed Clauses:**
**3.1** The site is intended for use solely for personal, non-commercial purposes. Users must use the site in accordance with the rules set out in the Agreement.

**3.5** Full or partial copying, reproduction, distribution, posting in public access on the Internet of the site's text and graphic materials, including design elements, as well as any use in the media and/or for commercial purposes without prior written permission from the Site Administration is prohibited.

**3.6** Copying, distributing, or posting in public access on the Internet data from the Summary section, which is not available to users without prior authorization, is prohibited.

**3.7** When posting data from the site in public access on the Internet, an active link to the information source (Source: "Weather and Climate" website) is required.

*(checked with google translate)*

#### **Ethics Summary**
**Robots.txt:** The rules for all user agents (`User-agent: *`) apply to my scraper. `/weather.php`, the only page I request for data, is not disallowed, and the file sets no crawl delay. I don't access any disallowed path, such as `/summary/`.

**Terms of use:**
- The site is for personal, non-commercial use only **(clause 3.1)**. This project is non-commercial coursework.
- The site's own text, graphics and design may not be copied or republished without permission **(clause 3.5)**, so the raw HTML pages stay private in my Google Drive, and only the extracted weather data goes on GitHub.
- Data from the members-only Summary section may not be shared **(clause 3.6)**. I don't use it.
- When the site's data is published online, a link to the source is required **(clause 3.7)**. The data is credited below and in the README.

**Limiting requests:** Each page is downloaded once and saved to Drive, and every later step reads the saved files, not the live site. Every download cell checks whether the page is already saved before sending a request, and there is a 3-second pause between requests, so the server never gets a burst of load.

**Necessary data only:** Only the two stations and the months needed for the analysis are requested. The pages contain public weather observations only; no login is used and no personal data is collected.

**Identifying the scraper:** Each request sends a User-Agent header that identifies it as a student project, rather than hiding what it is.

**The `bot` parameter:** The site's form asks whether the visitor is a robot, an alien or a human. Answering "robot" returns only the most recent week, whatever dates are requested (see 2.4), so the scraper answers "human" (`bot=2`, the form's default). Because this question acts as a check on automated visitors, I asked my lecturer whether this was acceptable, and they approved it on 23 October 2026.

**Source:** Weather data from [Pogoda i Klimat](http://www.pogodaiklimat.ru) (Источник: сайт «Погода и Климат»).

### **2.3 - Fetching and Inspecting One Page**
Before scraping every month, I look at one page in detail (occupied Donetsk, March 2024) to see how it is built. The site has one page per station per month, and the URL sets what is shown:

`http://www.pogodaiklimat.ru/weather.php?id=34519&bday=1&fday=31&amonth=3&ayear=2024&bot=2`

- `id`: the station number
- `bday` and `fday`: the first and last day to show (`fday` must be a real day, see 2.6)
- `amonth` and `ayear`: the month and year
- `bot`: the answer to the site's "robot / alien / human" question (see 2.4)

This section has three steps:

**1.** Fetch and save one page

**2.** Read and decode it

**3.** Find the tables that hold the readings

#### **2.3.1 - Fetching and Saving One Page**
I save the raw HTML to Drive straight away, so the page never needs downloading again. The cell first checks whether the file is already saved, so re-running it sends no request. `os.path.getsize()` gives the saved file's size in bytes.

In [26]:
import os  # for working with folders (already imported in setup; repeated so this cell works on its own)

# URL for one page: station 34519 (occupied Donetsk), days 1-31, March 2024
url = "http://www.pogodaiklimat.ru/weather.php?id=34519&bday=1&fday=31&amonth=3&ayear=2024&bot=2"
file_name = "raw_html/donetsk_2024_03.html"  # where the page is saved in Drive

os.makedirs("raw_html", exist_ok=True)  # create the folder if it isn't there yet; no error if it exists

if os.path.exists(file_name):  # already saved on an earlier run
    print("Already saved, not downloaded again")  # no request is sent
else:  # first run only
    request = urllib.request.Request(url)  # build the request
    request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
    raw_html = opener.open(request).read()  # send it and read the page as raw bytes
    f = open(file_name, "wb")  # open a new file for writing bytes
    f.write(raw_html)  # save the raw page to Drive
    f.close()  # close the file so it's fully saved
    print("Downloaded and saved")

print(os.path.getsize(file_name), "bytes")  # size of the saved file; first run gave 228,826

Already saved, not downloaded again
228826 bytes


#### **2.3.2 - Reading and Decoding the Page**
From here on I work from the saved file, not the live site. The file is stored as bytes, and the page is in Russian, so the bytes must be decoded with the right encoding. The page declares `charset=UTF-8` (shown in the output), and the Russian text prints correctly, so UTF-8 is the right choice.

In [27]:
f = open("raw_html/donetsk_2024_03.html", "rb")  # open the saved file for reading bytes ("rb")
saved_html = f.read()  # read the whole file as bytes
f.close()  # close the file

text = saved_html.decode("utf-8")  # turn the bytes into text using UTF-8
print(text[:500])  # the start of the page: the charset line should say UTF-8, and the Russian should be readable

<!DOCTYPE html>
<html lang="ru">
<head>
<title>Архив погоды в Донецке за последнюю неделю, погода в Донецке за март 2024 года</title>
    <meta http-equiv="content-type" content="text/html; charset=UTF-8">
    <meta name="description" content="Архив фактической погоды в Донецке (атмосферное давление, температура воздуха, влажность воздуха, направление и скорость ветра, атмосферные явления, осадки) за март 2024 года.">
    <meta name="keywords" content="архив, погода, в Донецке">
    <meta name="


#### **2.3.3 - Finding the Tables**
I load the page into BeautifulSoup and list every table, with its class and number of rows, to find which tables hold the 3-hourly readings.

In [31]:
soup = BeautifulSoup(text, "html.parser")  # turn the HTML text into a searchable object
tables = soup.find_all("table")  # list of every <table> on the page
print(len(tables), "tables")  # how many there are

for i in range(len(tables)):  # go through the tables by position
    rows = tables[i].find_all("tr")  # all the rows in this table
    print(i, "| class:", tables[i].get("class"), "|", len(rows), "rows")  # table number, class and row count
    print(tables[i].get_text()[:200])  # first 200 characters of its text
    print("-----")  # separator between tables

2 tables
0 | class: None | 243 rows


Время  (UTC),	дата


00
1.03


03
1.03


06
1.03


09
1.03


12
1.03


15
1.03


18
1.03


21
1.03


00
2.03


03
2.03


06
2.03


09
2.03


12
2.03


15
2.03


00
3.03


03
3.03


06
3.03


09
3.03
-----
1 | class: None | 243 rows













Ветер (напр.,м/с)
Видим.
Явления
Облачность
Т(С)
Тd(С)
f(%)
Тe(С)
Тes(С)
Комфортность
P(гПа)
Po(гПа)
Тmin(С)
Tmax(С)
R(мм)
R24(мм)
S(см)


В
2
10 км

ясно
-0.8
-4.3
77
-4
-4
по сезону
103
-----


#### **Checking the Cells in One Row**
Before writing the parser, I print every cell of the first reading. `repr()` shows the exact text, including empty cells (`''`), to confirm each row has 18 cells and to see what format the values are in.

In [32]:
weather_rows = tables[1].find_all("tr")  # all rows of the weather table (bot=2 page)
first_row_cells = weather_rows[1].find_all("td")  # the cells of the first data row (row 0 is the header)
print(len(first_row_cells), "cells")  # expect 18
for cell in first_row_cells:  # go through each cell
    print(repr(cell.get_text()))  # show its exact text; repr() shows hidden spaces too

18 cells
'В'
'2'
'10 км'
''
'ясно'
'-0.8'
'-4.3'
'77'
'-4'
'-4'
'по сезону'
'1032.0'
'1004.0'
''
''
''
''
''


### **2.4 - The Bot Parameter**
The page's search form asks whether the visitor is a robot, an alien or a human, and my URL contains `bot=2`. I find the form's `bot` field to see what each value means.

In [33]:
bot_inputs = soup.find_all("input", {"name": "bot"})  # choice buttons named "bot", if the form uses them
for item in bot_inputs:  # go through each one
    print(item, "|", item.next_sibling)  # the button itself and the text label after it

bot_select = soup.find("select", {"name": "bot"})  # a drop-down list named "bot", if the form uses one instead
if bot_select is not None:  # only run this if a drop-down was found
    for option in bot_select.find_all("option"):  # each choice in the list
        print(option.get("value"), "|", option.get_text())  # the value it sends, and its label
        #0 = робот (robot)
        #1 = инопланетянин (alien)
        #2 = человек (human), ticked by default

<input name="bot" type="radio" value="0"/> | робот
<input name="bot" type="radio" value="1"/> | инопланетянин
<input checked="checked" name="bot" type="radio" value="2"/> | человек


In [34]:
# The same page, but answering "robot" (bot=0) instead of "human" (bot=2)
test_url = "http://www.pogodaiklimat.ru/weather.php?id=34519&bday=1&fday=31&amonth=3&ayear=2024&bot=0"
test_file = "test_html/donetsk_2024_03_bot0.html"  # kept out of raw_html so it isn't counted as weather data

os.makedirs("test_html", exist_ok=True)  # create the test folder if needed

if os.path.exists(test_file):  # already saved on an earlier run
    f = open(test_file, "rb")  # open the saved copy
    test_bytes = f.read()  # read its bytes
    f.close()  # close the file
else:  # first run only
    request = urllib.request.Request(test_url)  # build the request
    request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
    test_bytes = opener.open(request).read()  # send it and read the raw bytes
    f = open(test_file, "wb")  # open a file for writing bytes
    f.write(test_bytes)  # save the test page
    f.close()  # close the file

print("bot=0:", len(test_bytes), "bytes | bot=2:", len(saved_html), "bytes")  # both in bytes, so they compare fairly

test_soup = BeautifulSoup(test_bytes.decode("utf-8"), "html.parser")  # make the bot=0 page searchable
test_rows = test_soup.find_all("table")[0].find_all("tr")  # rows of its time/date table
print(len(test_rows), "rows with bot=0 |", len(tables[0].find_all("tr")), "rows with bot=2")  # 53 vs 243 last time

first = test_rows[1].find_all("td")  # cells of the first reading (row 0 is the header)
last = test_rows[-1].find_all("td")  # cells of the last reading
print("bot=0 readings run from", first[1].get_text().strip(), first[0].get_text().strip(),
      "to", last[1].get_text().strip(), last[0].get_text().strip())  # date and UTC time of first and last reading

bot=0: 80856 bytes | bot=2: 228826 bytes
54 rows with bot=0 | 243 rows with bot=2
bot=0 readings run from 17.10 12 to 24.10 09


###**Bot Observarions**
#### **Observations on the bot Test**
**What each value means:** The form's three answers are 0 = robot (робот), 1 = alien (инопланетянин) and 2 = human (человек), with human ticked by default.

**Less data:** Answering "robot" returned 54 rows (1 header + 53 readings) and 80,856 bytes, against 243 rows and 228,826 bytes for the "human" page.

**The wrong dates:** I asked for March 2024, but the robot page's readings run from 17 October 12:00 to 24 October 09:00 (UTC), the week up to the day I ran the test. So with `bot=0` the site ignores the requested dates and only shows the latest week. This also explains why the same test gave 53 rows on 23 October and 54 on 24 October: it is a moving one-week window, not part of March.

**What this means:** With `bot=0`, no historical data can be reached at all, so the scraper has to use `bot=2`.


### **2.5 - Parsing One Month**
The page has two tables with the same number of rows: `tables[0]` holds the time and date, and `tables[1]` holds the 18 weather values. Row `i` in one belongs to row `i` in the other, so I go through them together, skipping the header row (row 0), and put each reading into a list. I keep the values as text for now and clean them in a later step.

In [35]:
time_rows = tables[0].find_all("tr")  # rows of the time/date table
weather_rows = tables[1].find_all("tr")  # rows of the weather table

# English names for the 20 columns: 2 from the time table + 18 from the weather table
column_names = ["time_utc", "date", "wind_dir", "wind_speed", "visibility", "phenomena", "cloud",
                "temp", "dew_point", "humidity", "temp_effective", "temp_effective_sun", "comfort",
                "pressure_sea", "pressure_station", "temp_min", "temp_max", "precip", "precip_24h", "snow_depth"]

rows = []  # will hold one list of 20 values per reading

for i in range(1, len(time_rows)):  # start at 1 to skip the header row
    row = []  # the values for this reading
    for cell in time_rows[i].find_all("td"):  # the time and date cells
        row.append(cell.get_text().strip())  # add the text, with spaces trimmed off the ends
    for cell in weather_rows[i].find_all("td"):  # the 18 weather cells in the matching row
        row.append(cell.get_text().strip())  # add each one
    rows.append(row)  # store this reading

for row in rows:  # safety check: every reading should have exactly 20 values
    if len(row) != 20:  # if one doesn't
        print("Unexpected row:", row)  # show it so I can see what went wrong

march = pd.DataFrame(rows, columns=column_names)  # turn the list of readings into a DataFrame
print(march.shape)  # expect (242, 20)
march.head(10)  # look at the first 10 readings

(242, 20)


,time_utc,date,wind_dir,wind_speed,visibility,phenomena,cloud,temp,dew_point,humidity,temp_effective,temp_effective_sun,comfort,pressure_sea,pressure_station,temp_min,temp_max,precip,precip_24h,snow_depth
0,00,1.03,В,2,10 км,,ясно,-0.8,-4.3,77,-4,-4,по сезону,1032.0,1004.0,,,,,
1,03,1.03,штиль,0,4000 м,дымка,7 баллов,-1.3,-3.6,84,-2,-2,тепло,1030.7,1002.7,-1.3,,,,
2,06,1.03,В,4,10 км,,9 баллов,-1.2,-3.7,83,-7,-7,по сезону,1030.7,1002.7,-1.4,,,,
3,09,1.03,В,4,20 км,,9 баллов,+2.7,-1.1,76,-2,-1,по сезону,1030.3,1002.7,,,,,
4,12,1.03,В,2,20 км,,7 баллов,+7.9,-1.6,51,+6,+8,тепло,1029.8,1002.7,,,,,
5,15,1.03,В,2,20 км,,7 баллов,+6.1,-2.0,56,+4,+4,тепло,1028.5,1001.3,,+8.1,,,
6,18,1.03,В,2,10 км,,6 баллов,+2.5,-2.6,69,-0,-0,тепло,1028.9,1001.3,,,,,
7,21,1.03,В,2,10 км,,7 баллов,+0.8,-2.1,81,-3,-3,по сезону,1029.0,1001.3,,,,,
8,00,2.03,В,4,10 км,,7 баллов,+0.3,-2.1,84,-5,-5,по сезону,1027.7,1000.0,,,,,
9,03,2.03,В,4,10 км,,9 баллов,-0.2,-1.9,88,-6,-6,по сезону,1026.5,998.7,-0.2,,,,


#### **Checking for Modelled Values**
The site says rows in italics are modelled estimates rather than real observations. I count `<i>` tags and the word "italic" in the page's HTML to see whether any rows are marked.

In [36]:
print(text.count("<i>"), text.count("italic"))  # does the page mark any rows in italics (modelled values)?

0 0


#### **Checking Wind Gusts**
Gusts appear in curly brackets after the average speed (e.g. `8 {14}`). I list every reading with a gust, count them, and list every different wind speed value, to see how gusts are recorded and what steps the station reports in.

In [37]:
gust_count = 0  # counts readings that have a gust

for i in range(len(march)):  # go through every reading
    if "{" in march["wind_speed"][i]:  # a curly bracket means a gust was recorded
        gust_count = gust_count + 1  # count it
        print(march["date"][i], march["time_utc"][i], march["wind_speed"][i])  # show when, and the value

print(gust_count, "readings with a gust")  # total, out of 242
print(march["wind_speed"].unique())  # every different value, to see what steps the station reports in

2.03 09 4 {10}
2.03 12 6 {10}
2.03 15 6 {12}
3.03 00 8 {14}
3.03 03 8 {14}
3.03 06 8 {14}
3.03 09 8 {14}
3.03 12 8 {14}
3.03 15 8 {14}
3.03 18 6 {14}
3.03 21 6 {12}
4.03 00 6 {12}
4.03 03 6 {12}
4.03 06 6 {12}
4.03 12 6 {12}
4.03 15 6 {12}
4.03 18 4 {12}
6.03 18 4 {10}
10.03 15 2 {10}
10.03 18 2 {10}
13.03 00 6 {10}
13.03 03 6 {10}
13.03 06 8 {14}
13.03 09 8 {18}
13.03 12 10 {19}
13.03 15 10 {20}
13.03 18 12 {20}
13.03 21 10 {20}
14.03 00 8 {20}
14.03 03 8 {16}
14.03 06 8 {16}
14.03 09 6 {12}
14.03 12 6 {12}
14.03 15 6 {10}
14.03 18 6 {10}
14.03 21 4 {10}
15.03 00 6 {10}
15.03 03 6 {10}
15.03 06 6 {10}
15.03 09 6 {10}
15.03 12 6 {10}
20.03 09 4 {10}
20.03 12 4 {10}
20.03 15 6 {10}
20.03 18 4 {10}
24.03 09 6 {10}
24.03 12 6 {10}
24.03 15 6 {10}
24.03 18 4 {10}
28.03 12 6 {10}
28.03 15 6 {10}
28.03 18 4 {10}
29.03 15 6 {11}
29.03 18 6 {12}
30.03 06 6 {10}
55 readings with a gust
['2' '0' '4' '4 {10}' '6 {10}' '6 {12}' '8 {14}' '6 {14}' '4 {12}'
 '2 {10}' '8 {18}' '10 {19}' '10 {20}' '12 

#### **Observations on the Parsed Month (March 2024)**
**Broken rows:** The parsed table has 242 readings and 20 columns. Every reading lined up with its time and date, and no row had the wrong number of values.

**Missing readings:** A full month would have 248 readings (31 days × 8). This month has 242, so 6 are missing. For example, 2 March has no 18:00 or 21:00 reading. Days with missing readings need a rule when readings are combined into days (2.11).

**Date and time:** The date only contains day.month (e.g. `2.03` is 2 March), with no year, so the year is added from the file name. Times are in UTC, and Donetsk is UTC+3, so the 21:00 UTC reading is midnight local time and belongs to the next local day. This is fixed before readings are grouped into days.

**Values:** Every value is stored as text (e.g. `+2.7`, `10 км`, `ясно`, `штиль`) and needs converting to numbers. Visibility is given in both kilometres and metres, so it will all be converted to metres.

**Blank cells:** A blank doesn't always mean missing data. A blank `phenomena` cell means nothing was happening (no fog, rain or snow). `temp_min` is only filled in at early-morning readings and `temp_max` at 15:00 UTC, so they are blank the rest of the time. Precipitation is also only filled in at some readings.

**Wind:**
- Gusts are shown in curly brackets after the average speed, e.g. `8 {14}`. 55 of the 242 readings have a gust, and the smallest gust is 10 m/s, so a blank gust probably means the gust was below 10 m/s rather than missing. This is checked on the full data in 2.10.
- The windiest spells were 3–4 March and 13–15 March, with gusts up to 20 m/s.
- Average wind speeds are always even numbers (0, 2, 4 … 12), so the station reports them in steps of 2 m/s, while gusts are given to the nearest 1 m/s. That makes the gust the more precise wind measure.

**Modelled values:** No italic markers were found in this month's HTML, so none of these readings is marked as a modelled estimate.

**Columns not needed:** `comfort`, `temp_effective` and `temp_effective_sun` are worked out by the site from the other values and aren't needed for the research questions, so they are dropped in cleaning (2.10).

### **2.6 - Testing Before the Full Scrape**
Before scraping every month, I test the pages most likely to behave differently from March 2024:
- **Kharkiv, March 2024:** checks the second station works.
- **Donetsk, April 2024:** a 30-day month, requested with `fday=31`, to see how the site handles a day that doesn't exist.
- **Donetsk, August 2023:** a month with known gaps, to see what a partly empty page looks like.

I also check whether 2022 has any data, to decide where the scrape should start.

Each page is saved to `raw_html`, with a 3-second pause between requests. `time.sleep()` pauses the code, and `.zfill(2)` adds a leading zero to the month (4 → "04") so the files sort in order. The cell skips any page already saved, so re-running it sends no requests.

In [38]:
import time  # for pausing between requests

# Each test page: [station id, station name for the file, year, month]
test_pages = [[34300, "kharkiv", 2024, 3],
              [34519, "donetsk", 2024, 4],
              [34519, "donetsk", 2023, 8]]

for page in test_pages:  # go through each test page
    station_id = page[0]  # the station number
    station_name = page[1]  # the name used in the file name
    year = page[2]  # the year
    month = page[3]  # the month

    file_name = "raw_html/" + station_name + "_" + str(year) + "_" + str(month).zfill(2) + ".html"  # e.g. raw_html/donetsk_2024_04.html

    if os.path.exists(file_name):  # already saved on an earlier run
        print(file_name, "already saved, not downloaded again |", os.path.getsize(file_name), "bytes")
        continue  # go to the next test page without sending a request

    # build the URL for this station and month (days 1-31, as in the original test)
    url = ("http://www.pogodaiklimat.ru/weather.php?id=" + str(station_id)
           + "&bday=1&fday=31&amonth=" + str(month) + "&ayear=" + str(year) + "&bot=2")

    request = urllib.request.Request(url)  # build the request
    request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
    raw_html = opener.open(request).read()  # send it and read the raw bytes

    f = open(file_name, "wb")  # open the file for writing bytes
    f.write(raw_html)  # save the page
    f.close()  # close the file

    print(file_name, len(raw_html), "bytes")  # what was saved and its size
    time.sleep(3)  # wait 3 seconds before the next request

raw_html/kharkiv_2024_03.html already saved, not downloaded again | 235823 bytes
raw_html/donetsk_2024_04.html already saved, not downloaded again | 220560 bytes
raw_html/donetsk_2023_08.html already saved, not downloaded again | 216644 bytes


#### **Checking the Test Pages**

In [39]:
test_files = ["raw_html/kharkiv_2024_03.html",
              "raw_html/donetsk_2024_04.html",
              "raw_html/donetsk_2023_08.html"]

for file_name in test_files:  # go through each saved test page
    f = open(file_name, "rb")  # open the saved file for reading bytes
    page_text = f.read().decode("utf-8")  # read it and turn the bytes into text
    f.close()  # close the file

    page_soup = BeautifulSoup(page_text, "html.parser")  # make it searchable
    page_tables = page_soup.find_all("table")  # find all the tables
    print(file_name, "|", len(page_tables), "tables")  # file name and number of tables

    if len(page_tables) >= 2:  # only check rows if both tables are there
        time_rows = page_tables[0].find_all("tr")  # rows of the time/date table
        weather_rows = page_tables[1].find_all("tr")  # rows of the weather table
        print("  rows:", len(time_rows), "and", len(weather_rows))  # should be the same number
        if len(time_rows) > 1:  # only if there is at least one reading after the header
            first = time_rows[1].find_all("td")  # cells of the first reading
            last = time_rows[-1].find_all("td")  # cells of the last reading ([-1] = last item)
            print("  first:", first[1].get_text().strip(), first[0].get_text().strip())  # date and time of first reading
            print("  last: ", last[1].get_text().strip(), last[0].get_text().strip())  # date and time of last reading

raw_html/kharkiv_2024_03.html | 2 tables
  rows: 249 and 249
  first: 1.03 00
  last:  31.03 21
raw_html/donetsk_2024_04.html | 2 tables
  rows: 233 and 233
  first: 1.04 00
  last:  30.04 21
raw_html/donetsk_2023_08.html | 2 tables
  rows: 231 and 231
  first: 2.08 06
  last:  31.08 21


#### **Retesting April with the Correct Last Day**

In [40]:
file_name = "raw_html/donetsk_2024_04.html"  # the April page

# Empty pages are about 38 KB and full months over 200 KB, so 100 KB separates them safely
if os.path.getsize(file_name) < 100000:  # the saved page is an empty one
    url = "http://www.pogodaiklimat.ru/weather.php?id=34519&bday=1&fday=30&amonth=4&ayear=2024&bot=2"  # fday=30, April's real last day
    request = urllib.request.Request(url)  # build the request
    request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
    raw_html = opener.open(request).read()  # send it and read the raw bytes
    f = open(file_name, "wb")  # overwrite the empty file
    f.write(raw_html)  # save the full page
    f.close()  # close the file
    print("Empty page replaced")
else:  # already the full month
    print("Saved page already has readings, not downloaded again")

f = open(file_name, "rb")  # open the saved page
april_soup = BeautifulSoup(f.read().decode("utf-8"), "html.parser")  # read, decode and make it searchable
f.close()  # close the file
april_rows = april_soup.find_all("table")[0].find_all("tr")  # rows of the time/date table
print(os.path.getsize(file_name), "bytes |", len(april_rows), "rows")  # first run gave 220,560 bytes, 233 rows

Saved page already has readings, not downloaded again
220560 bytes | 233 rows


#### **Checking the 2022 Coverage**
To decide where the scrape should start, I check whether the site has data for 2022. I request February and July 2022 for both stations (the start and end of the period that looked empty in earlier spot checks) and November 2022 for Donetsk, when its data seemed to begin. The pages are saved in `test_html`, so they aren't counted as weather data. *(This check was run after the main scrape, to confirm the start date I had chosen from the earlier spot checks.)*

In [41]:
# Pages to spot-check: [station id, name, year, month, last day of month]
check_pages = [[34519, "donetsk", 2022, 2, 28],
               [34300, "kharkiv", 2022, 2, 28],
               [34519, "donetsk", 2022, 7, 31],
               [34300, "kharkiv", 2022, 7, 31],
               [34519, "donetsk", 2022, 11, 30]]

os.makedirs("test_html", exist_ok=True)  # create the test folder if needed

for page in check_pages:  # go through each page to check
    file_name = "test_html/" + page[1] + "_" + str(page[2]) + "_" + str(page[3]).zfill(2) + ".html"  # e.g. test_html/donetsk_2022_02.html

    if not os.path.exists(file_name):  # only download if not saved on an earlier run
        url = ("http://www.pogodaiklimat.ru/weather.php?id=" + str(page[0])
               + "&bday=1&fday=" + str(page[4]) + "&amonth=" + str(page[3])
               + "&ayear=" + str(page[2]) + "&bot=2")  # URL with the month's real last day
        request = urllib.request.Request(url)  # build the request
        request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
        raw = opener.open(request).read()  # send it and read the raw bytes
        f = open(file_name, "wb")  # open a file for writing bytes
        f.write(raw)  # save the page
        f.close()  # close the file
        time.sleep(3)  # pause before the next request

    f = open(file_name, "rb")  # open the saved page
    page_soup = BeautifulSoup(f.read().decode("utf-8"), "html.parser")  # read, decode, make searchable
    f.close()  # close the file
    readings = page_soup.find_all("table")[0].find_all("tr")[1:]  # every reading, skipping the header row

    if len(readings) == 0:  # an empty page
        print(file_name, "| 0 readings")
    else:  # show how many readings and which dates they cover
        first_date = readings[0].find_all("td")[1].get_text().strip()  # date of the first reading
        last_date = readings[-1].find_all("td")[1].get_text().strip()  # date of the last reading
        print(file_name, "|", len(readings), "readings | first:", first_date, "| last:", last_date)

test_html/donetsk_2022_02.html | 0 readings
test_html/kharkiv_2022_02.html | 0 readings
test_html/donetsk_2022_07.html | 0 readings
test_html/kharkiv_2022_07.html | 0 readings
test_html/donetsk_2022_11.html | 85 readings | first: 18.11 | last: 30.11


#### **Observations on the Tests**
*The download cells now read saved files, so re-running them shows the final pages (for example, April now has data). The results below are from the first run on 23 October 2026.*

**Kharkiv, March 2024:** 235,823 bytes and 248 readings (1–31 March, none missing), so the second station works the same way as Donetsk.

**Donetsk, April 2024 with `fday=31`:** only 38,549 bytes, and the tables held only the header row, with no readings. The page still loaded normally, so nothing signalled an error. Requested again with `fday=30` (April's real last day), it returned 220,560 bytes and 232 readings. So **`fday` must be the month's real last day**, and an empty page can be spotted by its size (about 38 KB, compared with over 200 KB for a full month).

**Donetsk, August 2023:** 216,644 bytes and 230 readings, starting on 2 August at 06:00. The station's data restarts partway through the month, after a gap.

**2022:** I expected the site to have weather data for 2022, because the weather carries on regardless of the invasion. But what the site records is the station's reports, not the weather itself, and both stations have no readings at all in February or July 2022. The whole of February is empty, including the 23 days before the invasion began on 24 February, so the invasion alone doesn't explain the gap. The pages give no reason for it, so I can't say what caused it. Donetsk only resumes on 18 November 2022, and even then about 18% of readings are missing.

**What this means for the full scrape:** each request uses the month's real last day (including 29 for February 2024), and the scrape starts in January 2023 for both stations: from then on the data is far more complete, and using the same start date at both stations keeps them comparable.

### **2.7 - Scraping Every Month**
The tests in 2.6 showed that requesting a day that doesn't exist returns an empty page, so each request uses the month's real last day. February has 28 days, except in 2024, a leap year, when it has 29.

The loop goes through both stations and every month from January 2023 to October 2026 (92 pages). Each page is saved to `raw_html` before anything else is done with it. `os.path.exists()` checks whether a file is already saved; if it is (such as the test pages), the loop skips it, so no page is downloaded twice, and the loop can be re-run without repeating requests. There is a 3-second pause between requests.

When first run, the loop downloaded 88 pages and skipped the 4 already saved. The output below is from a later re-run, which confirms all 92 files exist and sends no requests.

In [42]:
# The two stations: [station id, name used in file names]
stations = [[34519, "donetsk"],
            [34300, "kharkiv"]]

years = [2023, 2024, 2025, 2026]  # years to scrape

days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]  # Jan to Dec; position 0 = January

downloaded = 0  # counts new pages fetched
skipped = 0  # counts pages already saved

for station in stations:  # go through each station
    station_id = station[0]  # the station number
    station_name = station[1]  # the name used in the file name

    for year in years:  # go through each year
        for month in range(1, 13):  # months 1 to 12
            if year == 2026 and month > 10:  # Kaggle data ends in October 2026
                continue  # skip November and December 2026

            last_day = days_in_month[month - 1]  # month 1 is position 0 in the list
            if year == 2024 and month == 2:  # 2024 is a leap year
                last_day = 29  # February 2024 has 29 days

            file_name = "raw_html/" + station_name + "_" + str(year) + "_" + str(month).zfill(2) + ".html"  # e.g. raw_html/kharkiv_2025_06.html

            if os.path.exists(file_name):  # already saved, e.g. one of the test pages
                skipped = skipped + 1  # count it
                continue  # move on without sending a request

            url = ("http://www.pogodaiklimat.ru/weather.php?id=" + str(station_id)
                   + "&bday=1&fday=" + str(last_day) + "&amonth=" + str(month)
                   + "&ayear=" + str(year) + "&bot=2")  # URL with this month's real last day

            request = urllib.request.Request(url)  # build the request
            request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
            response = opener.open(request)  # send it
            raw_html = response.read()  # the page as raw bytes

            f = open(file_name, "wb")  # open the file for writing bytes
            f.write(raw_html)  # save the page
            f.close()  # close the file

            downloaded = downloaded + 1  # count it
            print(file_name, len(raw_html), "bytes")  # show progress
            time.sleep(3)  # pause 3 seconds before the next request

print("Downloaded:", downloaded, "| Skipped (already saved):", skipped)  # first run: 88 downloaded, 4 skipped

Downloaded: 0 | Skipped (already saved): 92


#### **Re-fetching October 2026**
Both October 2026 pages came back empty (about 38 KB, the same size as the empty June–July 2023 pages). Day 31 is in the future, and like the 30-day months, the site seems to return nothing for a range it can't fill. I re-fetch October with day 20 as the last day, which covers the Kaggle data (to 10 October).

In [44]:
for station in stations:  # both stations, from the scraping cell
    file_name = "raw_html/" + station[1] + "_2026_10.html"  # the October 2026 page for this station

    if os.path.getsize(file_name) > 100000:  # already a page with readings (empty pages are about 38 KB)
        print(file_name, "already has readings, not downloaded again |", os.path.getsize(file_name), "bytes")
        continue  # go to the next station without sending a request

    url = ("http://www.pogodaiklimat.ru/weather.php?id=" + str(station[0])
           + "&bday=1&fday=20&amonth=10&ayear=2026&bot=2")  # October 2026, days 1-20

    request = urllib.request.Request(url)  # build the request
    request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
    raw_html = opener.open(request).read()  # send it and read the raw bytes

    f = open(file_name, "wb")  # replace the empty file
    f.write(raw_html)  # save
    f.close()  # close

    print(file_name, len(raw_html), "bytes")  # first run gave 163,485 and 166,835
    time.sleep(3)  # pause before the next request

raw_html/donetsk_2026_10.html already has readings, not downloaded again | 163485 bytes
raw_html/kharkiv_2026_10.html already has readings, not downloaded again | 166835 bytes


### **2.8 - Checking Coverage**
Before parsing every page, I check how complete the scraped data is. For each saved page I count the readings (rows minus the header) and the number of different days, to find months with gaps. `os.listdir()` lists the files in a folder, and `set()` keeps each date only once, so its length is the number of different days.

In [45]:
coverage_rows = []  # one entry per file

for file_name in sorted(os.listdir("raw_html")):  # every saved page, in name order
    f = open("raw_html/" + file_name, "rb")  # open the saved page
    page_soup = BeautifulSoup(f.read().decode("utf-8"), "html.parser")  # read, decode and make it searchable
    f.close()  # close the file

    time_rows = page_soup.find_all("table")[0].find_all("tr")  # rows of the time/date table
    dates = []  # the date of every reading
    for row in time_rows[1:]:  # skip the header row
        dates.append(row.find_all("td")[1].get_text().strip())  # second cell is the date

    parts = file_name.replace(".html", "").split("_")  # e.g. "donetsk_2024_04" -> ["donetsk", "2024", "04"]
    coverage_rows.append([parts[0], int(parts[1]), int(parts[2]), len(dates), len(set(dates))])  # station, year, month, readings, days

coverage = pd.DataFrame(coverage_rows, columns=["station", "year", "month", "readings", "days"])  # turn into a table
print(coverage.shape)  # expect (92, 5)
coverage[coverage["readings"] < 200]  # show only the months with fewer than 200 readings

(92, 5)


,station,year,month,readings,days
5,donetsk,2023,6,0,0
6,donetsk,2023,7,0,0
45,donetsk,2026,10,157,20
91,kharkiv,2026,10,160,20


#### **Missing Days and Readings per Month**

In [46]:
expected = []  # the number of days each month should have

for i in range(len(coverage)):  # go through every month in the table
    year = coverage["year"][i]  # this month's year
    month = coverage["month"][i]  # this month's number
    days = days_in_month[month - 1]  # normal length, from the scraping cell
    if year == 2024 and month == 2:  # leap year
        days = 29
    if year == 2026 and month == 10:  # only requested up to the 20th
        days = 20
    expected.append(days)  # store it

coverage["expected_days"] = expected  # add as a new column
coverage["missing_days"] = coverage["expected_days"] - coverage["days"]  # whole days with no readings
coverage["missing_readings"] = coverage["expected_days"] * 8 - coverage["readings"]  # readings missing in total

print(coverage.groupby("station")[["readings", "missing_days", "missing_readings"]].sum())  # totals for each station
coverage[coverage["missing_days"] > 0]  # months missing at least one whole day

         readings  missing_days  missing_readings
station                                          
donetsk     10351            63               761
kharkiv     11123             0               -11


,station,year,month,readings,days,expected_days,missing_days,missing_readings
5,donetsk,2023,6,0,0,30,30,240
6,donetsk,2023,7,0,0,31,31,248
7,donetsk,2023,8,230,30,31,1,18
13,donetsk,2024,2,210,28,29,1,22


#### **Investigating the Odd Months**
Kharkiv has 11 more readings than 8 per day allows, so I check which months have extra readings, which times they are at, and whether any date and time appears twice.

In [47]:
extra = coverage[coverage["missing_readings"] < 0]  # months with more readings than expected
print(extra[["station", "year", "month", "readings", "expected_days"]])  # show which months

for i in extra.index:  # go through each of those months
    file_name = ("raw_html/" + coverage["station"][i] + "_" + str(coverage["year"][i])
                 + "_" + str(coverage["month"][i]).zfill(2) + ".html")  # rebuild the file name
    f = open(file_name, "rb")  # open the saved page
    page_soup = BeautifulSoup(f.read().decode("utf-8"), "html.parser")  # read and make it searchable
    f.close()  # close the file

    times = []  # time of every reading
    pairs = []  # date and time of every reading together
    for row in page_soup.find_all("table")[0].find_all("tr")[1:]:  # every reading, skipping the header
        cells = row.find_all("td")  # the time and date cells
        times.append(cells[0].get_text().strip())  # the time
        pairs.append(cells[1].get_text().strip() + " " + cells[0].get_text().strip())  # e.g. "14.01 03"

    print(file_name)
    print(pd.Series(times).value_counts().sort_index())  # how many readings at each time; normal times are 00, 03 ... 21
    print("Duplicate date and time pairs:", len(pairs) - len(set(pairs)))  # 0 means no reading appears twice

    station  year  month  readings  expected_days
46  kharkiv  2023      1       250             31
50  kharkiv  2023      5       259             31
raw_html/kharkiv_2023_01.html
00    32
03    32
06    31
09    31
12    31
15    31
18    31
21    31
Name: count, dtype: int64
Duplicate date and time pairs: 2
raw_html/kharkiv_2023_05.html
00    31
03    31
06    31
09    33
12    37
15    32
18    31
21    33
Name: count, dtype: int64
Duplicate date and time pairs: 11


#### **Finding the Missing Day in February 2024**
The coverage table shows Donetsk is missing one whole day in February 2024. I list every date on the page and print any day from 1 to 29 that never appears.

In [48]:
f = open("raw_html/donetsk_2024_02.html", "rb")  # Donetsk, February 2024
page_soup = BeautifulSoup(f.read().decode("utf-8"), "html.parser")  # read and make it searchable
f.close()  # close the file

feb_dates = []  # dates found on the page
for row in page_soup.find_all("table")[0].find_all("tr")[1:]:  # every reading, skipping the header
    feb_dates.append(row.find_all("td")[1].get_text().strip())  # the date, e.g. "14.02"

for day in range(1, 30):  # days 1 to 29
    if str(day) + ".02" not in feb_dates:  # if this day never appears
        print("Missing:", str(day) + ".02")  # show it

Missing: 13.02


#### **Observations on the Scrape and Coverage**
**Coverage:** 92 pages were saved (46 months × 2 stations, January 2023 to 20 October 2026). At 8 readings a day, each station should have 1,389 days and 11,112 readings.

| Station | Readings | Missing days | Missing readings | Duplicates |
|---|---|---|---|---|
| Donetsk | 10,351 | 63 | 761 (about 7%) | not yet checked (see 2.10) |
| Kharkiv | 11,123 | 0 | 2 | 13 found so far |

**Donetsk:**
- June and July 2023 have no readings at all (61 days). This is the main gap in the weather data. June was requested with its correct last day (30), and other 30-day months requested the same way returned full data, so this is a real gap in the station's data and not a request error.
- Two other single days are missing: 1 August 2023 (the data restarts on the 2nd) and 13 February 2024. The pages give no reason.
- Outside the June–July gap, only about 2.6% of readings are missing, so the station is otherwise fairly complete.

**Kharkiv:**
- No whole days are missing, but the station has 11 more readings than 8 per day allows. Checking the two months responsible showed these are duplicates, not extra reports: every reading is at one of the normal times (00, 03 … 21), but some appear twice, 2 in January 2023 and 11 in May 2023.
- That makes 13 duplicates, while the total was only 11 too many, so Kharkiv is also missing 2 readings elsewhere. Adding up missing and extra readings hid both problems.
- This check only finds duplicates in months that have *more* readings than expected. A month with one duplicate and one missing reading would look normal, and Donetsk wasn't checked for duplicates at all. So 2.10 checks every reading at both stations.

**What this means for the next steps:**
- Duplicate readings must be removed before readings are combined into days, or they would be counted twice. 2.10 also checks whether the two copies of each duplicate have the same values.
- Days with no readings (the Donetsk gap, 1 August 2023 and 13 February 2024) will have no weather data after the merge. I leave them empty rather than filling them from Kharkiv, and use Kharkiv as a second station to check whether results hold.